# Processo de carga do relatório COLABORADORES e atualização do PBI HEADCOUNT

Tratamento da base COLAB, padronização de nomes de colunas e formatos de dados, além da atualização do PBI Headcount.

# Importação de Bibliotecas

In [2]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import duckdb as db
import shutil
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
import pyautogui
import pyperclip
import time
import os
import win32com.client

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [ ]:
# Configuração centralizada
CONFIG = {
    'id_processo': 7,
    'caminho_registros': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'diretorio_extracao': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\3. EXTRAÇÃO'),
    'diretorio_movidos': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'arquivo_final': Path(r'X:\Gestao_de_Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\COLABORADORES.xlsx'),
}

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Caminhos de pastas definidas')
print('')
print('----------------------------------------------------------------------------------------------------')

# Registra etapa do processo de forma segura

In [ ]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

# Retorna arquivos COLAB presentes no diretório

In [ ]:
def obter_arquivos_colaboradores(diretorio):
    if not diretorio.exists():
        print(f"❌ Diretório não existe: {diretorio}")
        return []
    
    lista_arquivos = [f.name for f in diretorio.glob('COLAB*.xlsx')]
    return lista_arquivos

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Arquivo COLAB na pasta')
print('')
print('----------------------------------------------------------------------------------------------------')

# Processa arquivo de colaboradores com DuckDB

In [ ]:
def processar_colaboradores(arquivo):
    df = pd.read_excel(arquivo)
    
    df_processado = db.query("""
        SELECT      
            REGISTRO AS registro,
            NOME_COMP AS nome,
            NASCTO AS nascimento,
            SEXO AS sexo,
            RACA_COR AS etnia_raca,
            FORMACAO AS cod_formacao,
            APOSENTADO AS aposentado,
            RG_NUMERO AS rg, 
            CPF_NUMERO AS cpf,
            DEFICIENTE AS deficiente,
            NACIONALI AS nacionalidade,
            "NAT.UF_ORI" AS naturalidade_uf,
            NAT_CIDADE AS naturalidade_cidade,
            EST_CIVIL AS estado_civil,
            SEXO_CONJU AS sexo_conjuge,
            NOME_CONJU AS nome_conjuge,
            DT_NAS_CON AS data_nasc_conjuge,
            CPF_CONJUG AS cpf_conjuge,
            FILHOS AS filhos,
            ADM_DATA AS data_admissao,
            SITU_DESCR AS situacao,
            RESC_DAT AS data_rescisao,
            RESC_DESCR AS descricao_rescisao,
            COD_EMPRES AS cod_empresa,            
            TAB_CARGO AS cod_cargo,
            CARGO_COMP AS cargo_completo,
            CARGO_DRES AS cargo_abreviado,
            TAB_CC_CON AS cod_centro_custo,
            NOM_GEST_C AS nome_gestor,
            TAB_SECAO AS cod_secao,
            SECAO AS secao,
            CCUSTO_CON AS centro_custo,
            SAL_ADMISS AS salario_admissao,
            SAL_ATUAL AS salario_atual,
            SAL_TOTAL AS salario_total,
            FGT_SALDO AS saldo_fgts,
            ENDERECO AS endereco,
            END_NUMERO AS numero_endereco,
            COMPLEMEN AS complemento,
            BAIRRO AS bairro,
            CIDADE AS cidade,
            ESTADO AS estado,
            TELEFONE1 AS telefone,
            E_MAIL AS email,
            SITUACAO AS cod_situacao,
            REG_TRABAL AS regime_trabalho,
            HOR_BASE AS hora_base,
            "HOR_COMPL." AS hora_complemento,
            PRO_DT_IND AS ultimo_reajuste_individual,
            PRO_DT_COL AS ultimo_reajuste_coletivo,
            FER_DT_INI AS data_inicio_ferias,
            FER_DT_FIM AS data_fim_ferias
        FROM (
            SELECT      
                *,
                ROW_NUMBER() OVER(  
                    PARTITION BY REGISTRO
                    ORDER BY SITUACAO ASC,
                    CASE WHEN RESC_DAT IS NULL THEN '2050-12-31' ELSE RESC_DAT END DESC
                ) AS ordem
            FROM df
        ) AS sub_query
        WHERE ordem = 1
    """).df()
    
    return df_processado

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Tratamento da base COLAB finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Cria arquivo Excel com formatação de tabela

In [ ]:
def criar_arquivo_excel(df, caminho, nome_tabela="COLABORADORES"):

    wb = Workbook()
    ws = wb.active
    ws.title = nome_tabela
    
    for r in dataframe_to_rows(df, index=False, header=True):
        ws.append(r)
    
    tabela = Table(displayName=nome_tabela, ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    wb.close()

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Arquivo COLABORADORES criado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Finalizando processo

In [ ]:
# Execução Principal
def main():
    config = CONFIG
    inicio = datetime.now()
    
    try:
        # Etapa 0
        append_registro_processo(config['caminho_registros'], config['id_processo'], 0)
        
        # Etapa 1 - Listar arquivos
        arquivos = obter_arquivos_colaboradores(config['diretorio_extracao'])
        
        if not arquivos:
            print("⚠️ Nenhum arquivo COLAB encontrado")
            return
        
        append_registro_processo(config['caminho_registros'], config['id_processo'], 1)
        
        # Processar arquivos
        for arquivo_nome in arquivos:
            try:
                arquivo_caminho = config['diretorio_extracao'] / arquivo_nome
                print(f"🔄 Processando: {arquivo_nome}")
                
                df_processado = processar_colaboradores(arquivo_caminho)
                criar_arquivo_excel(df_processado, config['arquivo_final'])
                
                # Mover arquivo
                destino = config['diretorio_movidos'] / arquivo_nome
                shutil.move(str(arquivo_caminho), str(destino))
                
                print(f"✅ {arquivo_nome} concluído")
                
            except Exception as e:
                print(f"❌ Erro ao processar {arquivo_nome}: {e}")
                continue
        
        # Etapa 8 - Finalização
        append_registro_processo(config['caminho_registros'], config['id_processo'], 8)
                
        duracao = datetime.now() - inicio
        
        print('—' * 100)
        print(f'\n   ✅ Processo finalizado com sucesso')
        print(f'   ⏱️  Tempo de execução: {duracao}\n')
        print('—' * 100)
        
    except Exception as e:
        print(f"❌ Erro crítico na execução: {e}")
        raise

if __name__ == "__main__":
    main()

# Atualização do Power BI - HEADCOUNT

In [ ]:
tempo_0 = [id, datetime.today(), 0]

pyautogui.PAUSE = 1

# Entrar no Power BI
path_pbi = r"X:\Gestao_de_Pessoas\Analytics\09 - Power BI\HEADCOUNT.pbix"
os.startfile(path_pbi) # Abre o arquivo pelo Windows
time.sleep(30)

# Atualizar Power BI
pyautogui.click(x=-830, y=103)
time.sleep(40)
pyautogui.click(x=-303, y=100)
time.sleep(5)
pyautogui.click(x=-745, y=473)
time.sleep(5)
pyautogui.click(x=-987, y=376)
pyautogui.press("tab")
pyautogui.press("enter")
time.sleep(5)
pyautogui.click(x=-758, y=575)
time.sleep(10)
pyautogui.click(x=-618, y=576)
time.sleep(5)
pyautogui.hotkey("alt", "F4")
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Power BI Atualizado')
print('')
print('----------------------------------------------------------------------------------------------------')

# Atualizando o arquivo Excel Controle_HC e Atestados
Caminho: X:\Gestao_de_Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos

In [ ]:
# Caminho do arquivo
path_excel = r"X:\Gestao_de_Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsb"
os.startfile(path_excel) # Abre o arquivo pelo Windows
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('HC!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'F5')
time.sleep(120)
pyautogui.hotkey('ctrl', 'b') # Salvando
time.sleep(60)
pyautogui.hotkey('alt', 'F4') # Fechando

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

Planilha atualizada com sucesso.


In [ ]:
tempo_1 = [id, datetime.today(), 8]

# Resumo de Finalização do Processo

print('----------------------------------------------------------------------------------------------------')
print('')
print('   Processo finalizado')
print('')
print('   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')